In [19]:
import matplotlib

# 不要强制 QtAgg：在无桌面/未装 Qt 时会导致 plot_* 卡住。
# - 在 Jupyter 内核里优先使用 inline 后端
# - 否则退回 Agg（不弹窗，适合服务器/无显示环境）
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None and "IPKernelApp" in ip.config:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    else:
        matplotlib.use("Agg")
except Exception:
    matplotlib.use("Agg")


In [20]:
import numpy as np
import mne
mne.set_log_level("INFO")  # INFO


In [21]:
set_path = r"./00002.set"
raw = mne.io.read_raw_eeglab(set_path, preload=True)
raw.info

Reading /home/johnsway/桌面/EEG_test/Dataset/2.fdt
Reading 0 ... 1206897  =      0.000 ...  4827.588 secs...


<Info | 8 non-empty values
 bads: []
 ch_names: FP1, FPz, FP2, AF7, AF3, AF4, AF8, F7, F5, F3, F1, Fz, F2, F4, ...
 chs: 65 EEG
 custom_ref_applied: False
 dig: 65 items (65 EEG)
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 65
 projs: []
 sfreq: 250.0 Hz
>

In [22]:
print(raw)
print("时长(s):", raw.times[-1])
print("采样率(Hz):", raw.info["sfreq"])
print("通道数:", len(raw.ch_names))

<RawEEGLAB | 2.fdt, 65 x 1206898 (4827.6 s), ~598.6 MiB, data loaded>
时长(s): 4827.588
采样率(Hz): 250.0
通道数: 65


In [23]:
tmax = min(5220, raw.times[-1])
raw.crop(tmin=0, tmax=tmax)
print("裁剪后时长(s):", raw.times[-1])

裁剪后时长(s): 4827.588


In [24]:
raw.plot(n_channels=32, duration=10, scalings="auto")

<MNEBrowseFigure size 800x800 with 4 Axes>

<MNEBrowseFigure size 800x800 with 4 Axes>

In [25]:
raw.set_channel_types({"DIGITAL IN": "stim"})
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, match_case=False)

<RawEEGLAB | 2.fdt, 65 x 1206898 (4827.6 s), ~598.6 MiB, data loaded>

In [26]:
print(raw.annotations)

<Annotations | 1386 segments: 1 (348), 2 (346), 3 (346), 4 (346)>


In [27]:
# 画传感器布局
raw.plot_sensors(show_names=True, show=True)

<Figure size 640x640 with 1 Axes>

<Figure size 640x640 with 1 Axes>

In [28]:
# 做平均参考
raw.set_eeg_reference("average", projection=False)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


<RawEEGLAB | 2.fdt, 65 x 1206898 (4827.6 s), ~598.6 MiB, data loaded>

In [29]:
#　频谱图 PSD　检查噪声/工频
raw.compute_psd(fmin=0.5, fmax=50).plot()

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


/home/johnsway/anaconda3/envs/pytorch/lib/python3.11/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNELineFigure size 1000x350 with 2 Axes>

In [30]:
# 复制一份用于对比
raw_filt = raw.copy()

In [31]:
# Notch
raw_filt.notch_filter(freqs=[50, 100], picks="eeg")
# Band-pass
raw_filt.filter(l_freq=1.0, h_freq=40.0, picks="eeg")

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 825 samples (3.300 s)



<RawEEGLAB | 2.fdt, 65 x 1206898 (4827.6 s), ~598.6 MiB, data loaded>

In [32]:
# PSD 对比
raw.compute_psd(fmin=0.5, fmax=50).plot()
raw_filt.compute_psd(fmin=0.5, fmax=50).plot()

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).
Effective window size : 8.192 (s)


/home/johnsway/anaconda3/envs/pytorch/lib/python3.11/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


Plotting power spectral density (dB=True).


/home/johnsway/anaconda3/envs/pytorch/lib/python3.11/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNELineFigure size 1000x350 with 2 Axes>

In [33]:
# 找坏道先用“看图 + 简单自动建议
raw_filt.plot(n_channels=32, duration=10, scalings="auto")

<MNELineFigure size 1000x350 with 2 Axes>

<MNELineFigure size 1000x350 with 2 Axes>

<MNELineFigure size 1000x350 with 2 Axes>

<MNEBrowseFigure size 800x800 with 4 Axes>

<MNEBrowseFigure size 800x800 with 4 Axes>

In [34]:
# # 如果发现明显飘、爆高噪声的通道，把它们加入 bads
# raw_filt.info["bads"] = ["Fp1"] 
# # 并做插值（可选，坏道少时很常用）
# raw_filt.interpolate_bads(reset_bads=False)

In [35]:
# 拟合 ICA
ica = mne.preprocessing.ICA(n_components=20, random_state=97, method="fastica")
ica.fit(raw_filt, picks="eeg")

Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by number: 20 components
Fitting ICA took 18.6s.


Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,52 iterations on raw data (1206898 samples)
ICA components,20
Available PCA components,64
Channel types,eeg
ICA components marked for exclusion,—


In [36]:
# 看 ICA 成分（成分拓扑）
import matplotlib.pyplot as plt
from IPython.display import display

print("开始绘图...")
figs = ica.plot_components(show=False)
print("绘图已生成，准备在 notebook 内嵌显示...")

if figs is None:
    figs = []
if not isinstance(figs, (list, tuple)):
    figs = [figs]

for fig in figs:
    display(fig)
    plt.close(fig)

开始绘图...
绘图已生成，准备在 notebook 内嵌显示...


<MNEFigure size 975x967 with 20 Axes>

In [37]:
# 看成分时间序列
ica.plot_properties(raw_filt, picks=[0, 1, 2])  # 先看前几个

    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
2413 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
2413 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
2413 matching events found
No baseline correction applied
0 projection items activated


<Figure size 700x600 with 6 Axes>

<Figure size 700x600 with 6 Axes>

<Figure size 700x600 with 6 Axes>

[<Figure size 700x600 with 6 Axes>,
 <Figure size 700x600 with 6 Axes>,
 <Figure size 700x600 with 6 Axes>]

In [38]:
# 应用 ICA（排除成分)
ica.exclude = [1, 2]  # 按观察改
raw_ica = raw_filt.copy()
ica.apply(raw_ica)

Applying ICA to Raw instance
    Transforming to ICA space (20 components)
    Zeroing out 2 ICA components
    Projecting back using 64 PCA components


<RawEEGLAB | 2.fdt, 65 x 1206898 (4827.6 s), ~598.6 MiB, data loaded>

In [39]:
# 再画 PSD 看看改善
raw_ica.compute_psd(fmin=0.5, fmax=50).plot()

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


/home/johnsway/anaconda3/envs/pytorch/lib/python3.11/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNELineFigure size 1000x350 with 2 Axes>

In [40]:
# 读取事件 events
events, event_id = mne.events_from_annotations(raw_ica)
print("event_id:", event_id)
print("events.shape:", events.shape)

Used Annotations descriptions: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4')]
event_id: {np.str_('1'): 1, np.str_('2'): 2, np.str_('3'): 3, np.str_('4'): 4}
events.shape: (1386, 3)


In [41]:
# 按“每个动作 6s”分段 Epoch
# 每个动作 6 s：常见是 事件开始后 0–6 s
# 如果你想包含一点基线，可用 tmin=-0.2, tmax=6.0 然后 baseline 用 -0.2–0。
tmin, tmax = 0.0, 6.0   # 先按 0-6s
baseline = None         # 如果 tmin=0 就不做 baseline；想做就改成 (-0.2, 0)

epochs = mne.Epochs(
    raw_ica,
    events=events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    preload=True,
    picks="eeg",
    reject_by_annotation=True
)
print(epochs)

Not setting metadata
1386 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1386 events and 1501 original time points ...
1 bad epochs dropped
<Epochs | 1385 events (all good), 0 – 6 s (baseline off), ~1015.2 MiB, data loaded,
 np.str_('1'): 347
 np.str_('2'): 346
 np.str_('3'): 346
 np.str_('4'): 346>


In [42]:
# Epochs 浏览
epochs.plot(n_channels=32, scalings="auto")

<MNELineFigure size 1000x350 with 2 Axes>

<MNEBrowseFigure size 800x800 with 4 Axes>

<MNEBrowseFigure size 800x800 with 4 Axes>

In [43]:
# 每个动作的平均波形 Evoked （ERP / ERD 的基础）
evokeds = {cond: epochs[cond].average() for cond in event_id.keys()}
list(evokeds.keys())

[np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4')]

In [44]:
# 画每个条件的全通道波形
for cond, evo in evokeds.items():
    evo.plot(window_title=cond, spatial_colors=True)

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

In [45]:
# 指定几个通道对比
# 看 C3/C4/Cz（运动相关常用）
mne.viz.plot_compare_evokeds(evokeds, picks=["C3", "Cz", "C4"])

combining channels using GFP (eeg channels)
combining channels using GFP (eeg channels)
combining channels using GFP (eeg channels)
combining channels using GFP (eeg channels)


<Figure size 800x600 with 1 Axes>

[<Figure size 800x600 with 1 Axes>]

In [46]:
# ERP 拓扑演化
for cond, evo in evokeds.items():
    evo.plot_topomap(times=np.linspace(0.2, 5.8, 8), ch_type="eeg", time_unit="s")

<MNEFigure size 1200x220 with 9 Axes>

<MNEFigure size 1200x220 with 9 Axes>

<MNEFigure size 1200x220 with 9 Axes>

<MNEFigure size 1200x220 with 9 Axes>

In [47]:
# 时频图（ERSP / ERD-ERS 常用）
freqs = np.linspace(4, 40, 30)
n_cycles = freqs / 2.0

cond0 = list(event_id.keys())[0]  # 先拿第一个条件示例
power = mne.time_frequency.tfr_morlet(
    epochs[cond0],
    freqs=freqs,
    n_cycles=n_cycles,
    return_itc=False,
    average=True
)

power.plot(picks=["C3"], baseline=None, mode=None, title=f"TFR - {cond0} - C3")

NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied


<Figure size 640x480 with 2 Axes>

[<Figure size 640x480 with 2 Axes>]

In [48]:
# 拓扑
power.plot_topomap(tmin=1.0, tmax=3.0, fmin=8, fmax=30, baseline=None, mode=None)

No baseline correction applied


<Figure size 200x200 with 2 Axes>

<Figure size 200x200 with 2 Axes>

In [49]:
# 质量检测
# 通道协方差/噪声结构
cov = mne.compute_raw_covariance(raw_ica, picks="eeg")
mne.viz.plot_cov(cov, raw_ica.info)
# GFP/全局场功率
for cond, evo in evokeds.items():
    evo.plot(gfp=True, titles=dict(eeg=f"GFP - {cond}"))

Using up to 24137 segments
Number of samples used : 1206850
[done]
Computing rank from covariance with rank=None
    Using tolerance 1.2e-11 (2.2e-16 eps * 64 dim * 8.1e+02  max singular value)
    Estimated rank (eeg): 61
    EEG: rank 61 computed from 64 data channels with 0 projectors


<Figure size 380x370 with 2 Axes>

<Figure size 380x370 with 1 Axes>

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

<Figure size 640x300 with 2 Axes>

In [50]:
# ———直接把 每个 trial 的 EEG 段导出为 .npz，供后续深度学习（PyTorch / TF）使用。———

In [51]:
sfreq = epochs.info["sfreq"]
n_times = len(epochs.times)
print("sfreq =", sfreq, "Hz")
print("n_times =", n_times, "samples")
print("epoch长度(s) =", n_times/sfreq)

sfreq = 250.0 Hz
n_times = 1501 samples
epoch长度(s) = 6.004


In [52]:
# X：形状默认是 (n_epochs, n_channels, n_times)
X = epochs.get_data()  # numpy array
print("X shape:", X.shape)  # (trial, ch, time)

X shape: (1385, 64, 1501)


In [53]:
# MNE 的 epoch 顺序与 epochs.events 对齐
y = epochs.events[:, 2].copy()   # 取 event code: 1,2,3,4
print("y unique:", np.unique(y, return_counts=True))

y unique: (array([1, 2, 3, 4]), array([347, 346, 346, 346]))


In [54]:
# 转成 0~3 符合常见 DL 分类习惯
y = y - 1
print("y unique (0-based):", np.unique(y, return_counts=True))

y unique (0-based): (array([0, 1, 2, 3]), array([347, 346, 346, 346]))


In [55]:
# 转 dtype：
X = X.astype(np.float32)
y = y.astype(np.int64)

In [56]:
# 按每个 trial 的每个通道做标准化
eps = 1e-8
mean = X.mean(axis=2, keepdims=True)
std  = X.std(axis=2, keepdims=True)
Xz = (X - mean) / (std + eps)

print("Xz mean/std check:", Xz.mean(), Xz.std())

Xz mean/std check: -5.311386e-11 0.9974125


In [57]:
# 导出为 npz（包含元信息）
out_path = "eeg_epochs_4class_0to6s.npz"

np.savez_compressed(
    out_path,
    X=Xz,              # (N, C, T) 标准化后的数据
    y=y,               # (N,) 0..3
    sfreq=sfreq,
    ch_names=np.array(epochs.ch_names),
    times=epochs.times.astype(np.float32),
    event_id=event_id,
    tmin=tmin,
    tmax=tmax
)

print("saved:", out_path)

saved: eeg_epochs_4class_0to6s.npz


In [58]:
# 验证读取
d = np.load(out_path, allow_pickle=True)
print(d["X"].shape, d["y"].shape)
print("sfreq:", d["sfreq"])
print("classes:", np.unique(d["y"], return_counts=True))
print("ch_names[:5]:", d["ch_names"][:5])

(1385, 64, 1501) (1385,)
sfreq: 250.0
classes: (array([0, 1, 2, 3]), array([347, 346, 346, 346]))
ch_names[:5]: ['FP1' 'FPz' 'FP2' 'AF7' 'AF3']
